### Create visualization for Affordable Panel Project - Task 1

by: lixi.liu@nrel.gov\
date: 12/21/23

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import re
from pathlib import Path
import pandas as pd

In [2]:
def bin_panel_sizes_every_50_amp(df_column):
    # left inclusive and right exclusive

    try:
        df_column = df_column.astype(float)
    except ValueError as e:
        df_column = df_column.replace('', np.nan).astype(float)

    df_out = pd.Series(np.nan, index=df_column.index)
    df_out.loc[df_column<50] = "<50"
    df_out.loc[(df_column>=50) & (df_column<100)] = "50-99"
    df_out.loc[(df_column>=100) & (df_column<150)] = "100-149"
    df_out.loc[(df_column>=150) & (df_column<200)] = "150-199"
    df_out.loc[(df_column>=200)] = "200+"

    categories = ["<50", "50-99", "100-149", "150-199", "200+"]
    df_out = pd.Categorical(df_out, ordered=True, categories=categories)

    return df_out

In [18]:
def _plot_bar(df, groupby_cols, output_dir=None, sfd_only=False):
    if sfd_only:
        cond = df["build_existing_model.geometry_building_type_recs"]=="Single-Family Detached"
        dfi = df.loc[cond, groupby_cols+["building_id"]]
    else:
         dfi = df[groupby_cols+["building_id"]]
    dfi = dfi.groupby(groupby_cols)["building_id"].count().unstack()

    fig, ax = plt.subplots()
    sort_index(dfi, axis=0).plot(kind="bar", ax=ax)
    if output_dir is not None:
        metric = "__by__".join(groupby_cols)
        fig.savefig(output_dir / f"bar_{metric}.png", dpi=400, bbox_inches="tight")
    plt.close()
    
def box_plot_by(df, panel_metric, by_var, output_dir=None, fig_file=None):
    ax = df[[by_var, panel_metric]].boxplot(by=by_var)

    if fig_file is None:
        fig_file = f"box_{panel_metric}_by_{by_var}.png"
    if output_dir is not None:
        fig_file = output_dir / fig_file
    ax.set_xlabel("Predicted Panel Amperage (Amp)")
    ax.set_ylabel("Existing Load (Amp)")
    plt.suptitle(None)
    if "83" in panel_metric:
        ax.set_title("Existing Load per NEC 220.83")
    else:
        ax.set_title("Existing Load per NEC 220.87 (Occupied Units Only)")
    ax.get_figure().savefig(fig_file, dpi=400, bbox_inches="tight")
    plt.close()

### Load files

In [4]:
nec_file = Path("/Users/lliu2/Documents/Documents_Files/Lab Call 5A - electrical panel constraints/FY23/Panels Estimation/euss1_2018_results_up00_clean__existing_load.csv")
pd_file = Path("/Users/lliu2/Documents/Documents_Files/Lab Call 5A - electrical panel constraints/FY23/Panels Estimation/euss1_2018_results_up00_clean__model_162__tsv_based__predicted_panels_probablistically_assigned.csv")

df = pd.read_csv(nec_file, low_memory=False, keep_default_na=False)
for col in ["existing_amp_220_83", "existing_amp_220_87"]:
    df[col] = df[col].replace('', np.nan).astype(float)

dfp = pd.read_csv(pd_file, low_memory=False, keep_default_na=False)
categories = ["<100", "100", "101-199", "200", "201+"]
dfp["predicted_panel_amp"] = pd.Categorical(dfp["predicted_panel_amp"], ordered=True, categories=categories)

df = df.join(dfp.set_index(["building_id"])["predicted_panel_amp"], on="building_id")
del dfp

df

,building_id,completed_status,build_existing_model.sample_weight,report_simulation_output.unmet_hours_cooling_hr,report_simulation_output.unmet_hours_heating_hr,build_existing_model.ahs_region,build_existing_model.ashrae_iecc_climate_zone_2004,build_existing_model.ashrae_iecc_climate_zone_2004_2_a_split,build_existing_model.bathroom_spot_vent_hour,build_existing_model.bedrooms,...,upgrade_costs.slab_perimeter_exposed_conditioned_ft,upgrade_costs.upgrade_cost_usd,upgrade_costs.wall_area_above_grade_conditioned_ft_2,upgrade_costs.wall_area_above_grade_exterior_ft_2,upgrade_costs.wall_area_below_grade_ft_2,upgrade_costs.window_area_ft_2,existing_load_total_VA,existing_amp_220_83,existing_amp_220_87,predicted_panel_amp
0,1,Success,242.131013,42.25,0.00,Non-CBSA South Atlantic,4A,4A,Hour22,3,...,85.3,0.0,2133.4,2507.4,0.0,332.3,23999.714653,99.998811,37.552083,200
1,2,Success,242.131013,0.00,77.25,Non-CBSA Mountain,6B,6B,Hour22,3,...,121.3,0.0,970.6,1088.0,970.6,116.4,14942.300000,62.259583,35.052083,<100
2,4,Success,242.131013,0.00,0.00,Non-CBSA West South Central,2A,"2A - TX, LA",Hour7,4,...,0.0,0.0,1557.8,1860.0,0.0,187.0,29707.416345,123.780901,NaN,200
3,5,Success,242.131013,163.50,2.25,Non-CBSA West North Central,4A,4A,Hour19,3,...,171.2,0.0,1657.6,2263.8,0.0,123.3,22739.467573,94.747782,97.187500,200
4,6,Success,242.131013,0.00,0.00,"CBSA Miami-Fort Lauderdale-West Palm Beach, FL",2A,"2A - FL, GA, AL, MS",Hour7,3,...,0.0,0.0,1485.2,2024.0,742.6,143.6,17529.440000,73.039333,32.135417,200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548911,549996,Success,242.131013,0.00,0.00,"CBSA Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",4A,4A,Hour0,2,...,0.0,0.0,975.4,975.4,487.6,146.2,16552.685206,68.969522,34.270833,200
548912,549997,Success,242.131013,234.00,0.00,"CBSA New York-Newark-Jersey City, NY-NJ-PA",5A,5A,Hour8,3,...,0.0,0.0,912.8,912.8,912.8,109.5,13747.292788,57.280387,8.854167,200
548913,549998,Success,242.131013,142.75,0.00,"CBSA Los Angeles-Long Beach-Anaheim, CA",3B,3B,Hour3,2,...,154.8,0.0,1430.0,1894.8,0.0,222.9,18833.549667,78.473124,48.802083,101-199
548914,549999,Success,242.131013,32.50,20.75,Non-CBSA East South Central,3A,3A,Hour17,5,...,0.0,0.0,1657.6,2263.8,828.8,205.4,20563.814670,85.682561,50.156250,200


In [5]:
sfd_only = False
plot_dir_name = "plots_sfd" if sfd_only else "plots"
output_dir = nec_file.parent / f"{plot_dir_name}__existing_load" # <---
output_dir.mkdir(parents=True, exist_ok=True) 
print(f"Plots are outputing to: {output_dir}")

Plots are outputing to: /Users/lliu2/Documents/Documents_Files/Lab Call 5A - electrical panel constraints/FY23/Panels Estimation/plots__existing_load


#### Box plots

In [6]:
panel_metric = "existing_amp_220_83"
by_var = "predicted_panel_amp"
box_plot_by(df, panel_metric, by_var, output_dir=output_dir)

panel_metric = "existing_amp_220_87"
by_var = "predicted_panel_amp"
box_plot_by(df, panel_metric, by_var, output_dir=output_dir)

In [20]:
dfe = df.loc[df["build_existing_model.heating_fuel"]=="Electricity"]
panel_metric = "existing_amp_220_83"
by_var = "predicted_panel_amp"
box_plot_by(dfe, panel_metric, by_var, output_dir=output_dir, 
            fig_file=f"box_plot_electricity_heat_fuel_{panel_metric}")

panel_metric = "existing_amp_220_87"
by_var = "predicted_panel_amp"
box_plot_by(dfe, panel_metric, by_var, output_dir=output_dir, 
            fig_file=f"box_plot_electricity_heat_fuel_{panel_metric}")


#### Outliers

In [24]:
def outliers_in_box_plot(df, panel_metric):
    dfi = df.loc[~df[panel_metric].isna()]
    ub = dfi.groupby(["predicted_panel_amp"])[panel_metric].apply(
        lambda x: np.percentile(x, 75)+ (np.percentile(x, 75)-np.percentile(x, 25))*1.5
    )

    print(f"For {panel_metric}, there are {len(dfi)} / {len(df)} datapoints...")
    for pa in ub.index:
        n = len(dfi[dfi["predicted_panel_amp"]==pa])
        n_out = len(dfi[(dfi["predicted_panel_amp"]==pa) & (dfi[panel_metric]>ub[pa])])
        print(f" - for {pa} A, {n_out} models  ({n_out/n*100:.02f}%)  are outliers larger than UB: {ub[pa]}")
        

def data_within_bound_in_box_plot(df, panel_metric):
    ub = {
        "<100": 75,
        "100": 100,
        "101-199": 175,
        "200": 200,
        "201+": 300,
    }
    dfi = df.loc[~df[panel_metric].isna()]

    print(f"For {panel_metric}, there are {len(dfi)} / {len(df)} datapoints...")
    for pa in ub.keys():
        cond1 = (dfi["predicted_panel_amp"]==pa)
        cond2 = cond1 & (dfi[panel_metric]<=ub[pa])
        n = len(dfi.loc[cond1])
        n_in = len(dfi.loc[cond2])
        print(f" - for {pa} A predicted panels, {n_in} models  ({n_in/n*100:.02f}%)  are within the bound: {ub[pa]}")

In [23]:
panel_metric = "existing_amp_220_83"
outliers_in_box_plot(df, panel_metric)

print()
panel_metric = "existing_amp_220_87"
outliers_in_box_plot(df, panel_metric)

For existing_amp_220_83, there are 548916 / 548916 datapoints...
 - for <100 A, 491 models  (3.51%)  are outliers larger than UB: 116.36477155073143
 - for 100 A, 3145 models  (3.10%)  are outliers larger than UB: 108.89335452731311
 - for 101-199 A, 3510 models  (3.44%)  are outliers larger than UB: 110.70274601903175
 - for 200 A, 9284 models  (3.01%)  are outliers larger than UB: 137.5663650212513
 - for 201+ A, 816 models  (3.61%)  are outliers larger than UB: 131.20298092316148

For existing_amp_220_87, there are 482597 / 548916 datapoints...
 - for <100 A, 656 models  (5.29%)  are outliers larger than UB: 94.08854166666669
 - for 100 A, 4331 models  (4.79%)  are outliers larger than UB: 85.05208333333331
 - for 101-199 A, 4593 models  (5.09%)  are outliers larger than UB: 93.85416666666669
 - for 200 A, 10745 models  (3.98%)  are outliers larger than UB: 135.78125
 - for 201+ A, 1052 models  (5.28%)  are outliers larger than UB: 108.86067708333333


In [27]:
### electric heat fuel only
panel_metric = "existing_amp_220_83"
outliers_in_box_plot(dfe, panel_metric)

print()
panel_metric = "existing_amp_220_87"
outliers_in_box_plot(dfe, panel_metric)

For existing_amp_220_83, there are 220285 / 220285 datapoints...
 - for <100 A, 91 models  (2.83%)  are outliers larger than UB: 137.4404701446827
 - for 100 A, 317 models  (2.22%)  are outliers larger than UB: 132.30862730871456
 - for 101-199 A, 883 models  (2.91%)  are outliers larger than UB: 124.70905985190808
 - for 200 A, 5474 models  (3.32%)  are outliers larger than UB: 146.08524812952663
 - for 201+ A, 256 models  (3.30%)  are outliers larger than UB: 149.0428545011825

For existing_amp_220_87, there are 189552 / 220285 datapoints...
 - for <100 A, 78 models  (2.83%)  are outliers larger than UB: 160.22135416666669
 - for 100 A, 346 models  (2.81%)  are outliers larger than UB: 153.60026041666669
 - for 101-199 A, 848 models  (3.26%)  are outliers larger than UB: 142.94270833333331
 - for 200 A, 4704 models  (3.32%)  are outliers larger than UB: 165.33854166666666
 - for 201+ A, 230 models  (3.43%)  are outliers larger than UB: 163.50911458333331


In [ ]:
### electric heat fuel only
panel_metric = "existing_amp_220_83"
outliers_in_box_plot(dfe, panel_metric)

print()
panel_metric = "existing_amp_220_87"
outliers_in_box_plot(dfe, panel_metric)

#### Number within acceptable bound per panel label

In [25]:
panel_metric = "existing_amp_220_83"
data_within_bound_in_box_plot(df, panel_metric)

print()
panel_metric = "existing_amp_220_87"
data_within_bound_in_box_plot(df, panel_metric)

For existing_amp_220_83, there are 548916 / 548916 datapoints...
 - for <100 A predicted panels, 9019 models  (64.56%)  are within the bound: 75
 - for 100 A predicted panels, 95933 models  (94.57%)  are within the bound: 100
 - for 101-199 A predicted panels, 101905 models  (99.87%)  are within the bound: 175
 - for 200 A predicted panels, 308252 models  (99.79%)  are within the bound: 200
 - for 201+ A predicted panels, 22578 models  (99.99%)  are within the bound: 300

For existing_amp_220_87, there are 482597 / 548916 datapoints...
 - for <100 A predicted panels, 11178 models  (90.12%)  are within the bound: 75
 - for 100 A predicted panels, 87741 models  (97.12%)  are within the bound: 100
 - for 101-199 A predicted panels, 89855 models  (99.57%)  are within the bound: 175
 - for 200 A predicted panels, 267670 models  (99.25%)  are within the bound: 200
 - for 201+ A predicted panels, 19902 models  (99.91%)  are within the bound: 300


In [26]:
### electric heat fuel only
panel_metric = "existing_amp_220_83"
data_within_bound_in_box_plot(dfe, panel_metric)

print()
panel_metric = "existing_amp_220_87"
data_within_bound_in_box_plot(dfe, panel_metric)

For existing_amp_220_83, there are 220285 / 220285 datapoints...
 - for <100 A predicted panels, 1420 models  (44.20%)  are within the bound: 75
 - for 100 A predicted panels, 12059 models  (84.45%)  are within the bound: 100
 - for 101-199 A predicted panels, 30201 models  (99.69%)  are within the bound: 175
 - for 200 A predicted panels, 164163 models  (99.66%)  are within the bound: 200
 - for 201+ A predicted panels, 7764 models  (99.97%)  are within the bound: 300

For existing_amp_220_87, there are 189552 / 220285 datapoints...
 - for <100 A predicted panels, 1870 models  (67.88%)  are within the bound: 75
 - for 100 A predicted panels, 10445 models  (84.89%)  are within the bound: 100
 - for 101-199 A predicted panels, 25659 models  (98.71%)  are within the bound: 175
 - for 200 A predicted panels, 139832 models  (98.61%)  are within the bound: 200
 - for 201+ A predicted panels, 6680 models  (99.73%)  are within the bound: 300


In [12]:
(548916-482597)/548916

0.12081812153407807

#### Binned histograms

In [ ]:
### bin panels
df["binned_existing_amp_220_83"]= bin_panel_sizes_every_50_amp(df["existing_amp_220_83"])
df["binned_existing_amp_220_87"]= bin_panel_sizes_every_50_amp(df["existing_amp_220_87"])

# save a copy
df.to_csv(output_dir / "euss1_2018_results_up00_clean__model_162__tsv_based__vs__existing_load.csv", index=False)

print("Binned existing amps")

In [ ]:
panel_metric = "binned_existing_amp_220_83"
_plot_bar(df, ["predicted_panel_amp", panel_metric], output_dir=output_dir, sfd_only=sfd_only)

panel_metric = "binned_existing_amp_220_87"
_plot_bar(df, ["predicted_panel_amp", panel_metric], output_dir=output_dir, sfd_only=sfd_only)